# Database Space Management: ML Forecasting Pipeline
This notebook extracts historical drive space telemetry from MSSQL and uses Machine Learning to forecast when servers will drop below 30% free space.

In [23]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
import warnings
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
import plotly.express as px
import os
from sqlalchemy import text # needed for the insert statement in the write_anomalies_to_db function

#new feature request for database drive letter growth monitor.  This will ensure the model to make smarter predictions on the drive letter growth and help to identify potential issues before they become critical.
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
plt.style.use("fivethirtyeight")

# Create output directory for charts if it doesn't exist
os.makedirs('./output', exist_ok=True)

## 1. Data Ingestion
Connect to the local MSSQL database and retrieve the daily growth summary.

In [24]:
# Make a DataFrame
# engine = create_engine('mssql+pyodbc://localhost/DBA?driver=SQL+Server+Native+Client+18.0')
engine = create_engine('mssql+pyodbc://localhost/DBA?driver=ODBC+Driver+18+for+SQL+Server&TrustServerCertificate=yes'
                       ,isolation_level="AUTOCOMMIT"
                       )

query = '''
SELECT [DTINSERTED], dbserver, [DriveLetter], [Previous Space], [Current Space], 
       [Growth], [TotalDriveSize], [PercentageFree],
       CASE WHEN [PercentageFree] < 30 THEN 1 ELSE 0 END as LowPercentFree 
FROM [dba].[dbo].[MonthGrowthSummary] WITH(NOLOCK) 
ORDER BY CONVERT(char(10), [DTINSERTED], 121) ASC
'''

growth_size = pd.read_sql(query, engine)
growth_size.head()

,DTINSERTED,dbserver,DriveLetter,Previous Space,Current Space,Growth,TotalDriveSize,PercentageFree,LowPercentFree
0,2021-01-01,SQLSAMPLE-0001,C,448182,448097,85,512000,87.519,0
1,2021-01-01,SQLSAMPLE-0001,D,80880,80880,0,102400,78.984,0
2,2021-01-01,SQLSAMPLE-0001,E,501229,501229,0,1048576,47.801,0
3,2021-01-01,SQLSAMPLE-0001,L,46078,46078,0,102400,44.998,0
4,2021-01-01,SQLSAMPLE-0002,C,85308,85308,0,102400,83.309,0


## 1.A Notes

In [25]:
# import pyodbc
# drivers = [d for d in pyodbc.drivers()]
# print("Available ODBC Drivers:")
# for d in drivers:
#     print(d)
# get current drivers for MSSQL Server

In [26]:
# !pip install -U kaleido
# #installed missingr library for plotly to save charts as images

In [27]:
# import os
# os._exit(0)

## 1.1 Making use of K-Means to help the model to make better predictions
It will analyze the drives, assign them a DriveProfileCluster (0, 1, or 2), and attach that as a new column

In [28]:
growth_size = pd.read_sql(query, engine)


def profile_drives_with_kmeans(df):
    print("Applying K-Means to profile drive behavior...")
    
    features = ['TotalDriveSize', 'Growth', 'PercentageFree']
    X = df[features].copy().fillna(0)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    
    df['DriveProfileCluster'] = kmeans.fit_predict(X_scaled)
    
    distances = kmeans.transform(X_scaled)
    
    # 2. Find the distance to its *assigned* cluster center
    min_distances = distances.min(axis=1)
    df['DistanceToCentroid'] = min_distances
    
    # 3. Define the Anomaly Threshold (e.g., the top 1% most extreme distances)
    # Alternatively, you could use a hard Z-score or fixed number.
    threshold = np.percentile(min_distances, 99) 
    
    # --- MISSING POISON PILL LOGIC (Add this) ---
    # Count how many drives are in each cluster. If a cluster has less than 5 records, it's anomalous.
    cluster_counts = df['DriveProfileCluster'].value_counts()
    fringe_clusters = cluster_counts[cluster_counts < 5].index

    # 4. Flag the anomalies
    # df['IsAnomaly'] = df['DistanceToCentroid'] > threshold
    df['IsAnomaly'] = (df['DistanceToCentroid'] > threshold) | (df['DriveProfileCluster'].isin(fringe_clusters))
    
    print(f"Anomaly threshold set at distance: {threshold:.2f}")
    print(f"Identified {df['IsAnomaly'].sum()} anomalies out of {len(df)} records.")

    # df_centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
    # df_centroids.index.name = 'Cluster'
    # print("\n--- Cluster Profiles (Average Values) ---")
    # print(df_centroids.round(2))
    
    return df


growth_size = profile_drives_with_kmeans(growth_size)

# # ==========================================
# # 3. Proceed to Preprocessing
# # ==========================================
# growth_size_processed = preprocess_data(growth_size.copy())

Applying K-Means to profile drive behavior...
Anomaly threshold set at distance: 2.04
Identified 1851 anomalies out of 185089 records.


## 1.2 Write the anomalies found in data to a table called anomaliesfoundML
This will pull in the anamalies that it found and insert into the table

In [29]:
def write_anomalies_to_db(anomaly_df, db_engine):
    if anomaly_df.empty:
        print("No anomalies detected. Proceeding normally.")
        return # Do nothing if no anomalies were found

    print(f"Writing {len(anomaly_df)} anomalies to the 'anomaliesfoundML' table...")

    # Select the columns that are useful for the DBA team to review later
    columns_to_save = [
        'dbserver', 'DriveLetter','Previous Space', 
        'Current Space', 'Growth', 'TotalDriveSize', 'PercentageFree', 
        'DriveProfileCluster', 'DistanceToCentroid'
    ]
    
    # Filter the dataframe to just those columns
    df_to_save = anomaly_df[columns_to_save].copy()

    
    
    try:
        # Write directly to the SQL Server table
        # if_exists='append' ensures we add to the historical log without deleting old records
        with db_engine.begin() as connection:  # Use a transaction context manager
            df_to_save.to_sql(
                name='anomaliesfoundML_stg', 
                con=connection, 
                schema='dbo', # Update this if your schema is different
                if_exists='append', 
                index=False
            )
        
            merge_sql = """
            INSERT INTO [dba].[dbo].[anomaliesfoundML] 
            ([dbserver], [DriveLetter], [Previous Space], 
             [Current Space], [Growth], [TotalDriveSize], [PercentageFree], 
             [DriveProfileCluster], [DistanceToCentroid])
             
            SELECT s.[dbserver], s.[DriveLetter], s.[Previous Space], 
                   s.[Current Space], s.[Growth], s.[TotalDriveSize], s.[PercentageFree], 
                   s.[DriveProfileCluster], s.[DistanceToCentroid]
            FROM [dba].[dbo].[anomaliesfoundML_stg] s
            WHERE NOT EXISTS (
                SELECT 1 
                FROM [dba].[dbo].[anomaliesfoundML] target
                WHERE target.dbserver = s.dbserver 
                  AND target.DriveLetter = s.DriveLetter
                  AND target.DTINSERTED = s.DTINSERTED
            );
            """
            result = connection.execute(text(merge_sql))
            rows_inserted = result.rowcount
            
            connection.execute(text("TRUNCATE TABLE [dba].[dbo].[anomaliesfoundML_stg];"))

        print("Successfully logged anomalies to the database and truncated the staging table.")
    except Exception as e:
        print(f"Failed to write anomalies to database: {e}")


# ==========================================
# 2 & 3. Execute Logging & Purge Anomalies (Bulletproofed)
# ==========================================

# Check if the column exists before trying to filter or drop it
if 'IsAnomaly' in growth_size.columns:
    # Extract anomalies
    anomalies_only = growth_size[growth_size['IsAnomaly'] == True]

    # Write to DB
    write_anomalies_to_db(anomalies_only, engine) 

    # Overwrite growth_size to ONLY keep normal data
    growth_size = growth_size[growth_size['IsAnomaly'] == False].copy()

    # Drop the temporary Anomaly flags
    growth_size = growth_size.drop(columns=['DistanceToCentroid', 'IsAnomaly'])
    print("✅ Anomalies purged from training data.")
else:
    print("⚠️ Anomalies have already been processed and dropped from this dataframe.")

Writing 1851 anomalies to the 'anomaliesfoundML' table...
Successfully logged anomalies to the database and truncated the staging table.
✅ Anomalies purged from training data.


## 2. Feature Engineering & Preprocessing
Convert timestamps, extract date components, and encode categorical variables. 
*Note: We purposefully drop current/previous space metrics to prevent data leakage during forecasting.*

In [30]:
def preprocess_data(df):
    df["DTINSERTED"] = pd.to_datetime(df["DTINSERTED"], errors="coerce")
    
    # Preserve original text labels
    df["dbserver_text"] = df["dbserver"]
    df["DriveLetter_text"] = df["DriveLetter"]

    # Extract date features (FIXED: dayofyear)
    df["DateFreeInAYear"] = df.DTINSERTED.dt.year
    df["DateFreeInAMonth"] = df.DTINSERTED.dt.month
    df["DateFreeInADay"] = df.DTINSERTED.dt.day
    df["DateFreeInDayOfWeek"] = df.DTINSERTED.dt.dayofweek
    df["DateFreeInDayOfYear"] = df.DTINSERTED.dt.dayofyear

    for label, content in df.items():
        if label in ["dbserver_text", "DriveLetter_text"]:
            continue

        if pd.api.types.is_string_dtype(content):
            df[label] = content.astype("category").cat.as_ordered()
            
        if pd.api.types.is_numeric_dtype(content):
            if pd.isnull(content).sum():
                df[label+"_is_missing"] = pd.isnull(content)
                df[label] = content.fillna(content.median())

        if not pd.api.types.is_numeric_dtype(content):
            df[label+"_is_missing"] = pd.isnull(content)
            df[label] = pd.Categorical(content).codes + 1
            
    return df

growth_size_processed = preprocess_data(growth_size.copy())

# Recreate Date Index for Time-Series logic
growth_size_processed["date"] = pd.to_datetime(dict(
    year=growth_size_processed.DateFreeInAYear, 
    month=growth_size_processed.DateFreeInAMonth, 
    day=growth_size_processed.DateFreeInADay
))
dataframe = growth_size_processed.set_index("date")

## 3. Global Random Forest Model (Baseline)
Train a baseline Random Forest Regressor to test global predictive accuracy.

In [31]:
np.random.seed(42)

# Prevent Data Leakage: Drop columns that mathmatically equal PercentageFree
columns_to_drop = ["PercentageFree", "DTINSERTED_is_missing", "dbserver_is_missing", 
                   "DriveLetter_is_missing", "Current Space", "Previous Space", 
                   "Growth", "TotalDriveSize", "LowPercentFree"]

tempX = growth_size_processed.drop(columns=columns_to_drop, errors='ignore')
X = tempX.set_index("date")
y = growth_size_processed["PercentageFree"]

X_numeric = X.select_dtypes(include=[np.number])

# Time-Series Split (Do not use random split for time series)
split_idx = int(len(X_numeric) * 0.8)
X_train, X_test = X_numeric.iloc[:split_idx], X_numeric.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

rfr = RandomForestRegressor(n_estimators=100)
rfr.fit(X_train, y_train)

y_preds = rfr.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test, y_preds):.2f}")
print(f"R2 Score: {r2_score(y_test, y_preds):.2f}")

MAE: 1.04
R2 Score: 0.98


## 4. Server-Specific Linear Regression Forecasting
Forecast the next 30, 60, 90, and 365 days for *each individual server and drive combination*.

In [32]:
forecast_results = []
horizons = [30, 60, 90, 365]

for (server_code, drive_code), group in growth_size_processed.groupby(["dbserver", "DriveLetter"]):
    group = group.sort_values("date")

    X_lin = group["date"].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
    y_lin = group["PercentageFree"].values

    model = LinearRegression()
    model.fit(X_lin, y_lin)

    for horizon in horizons:
        future_dates = pd.date_range(start=group["date"].max() + pd.Timedelta(days=1), periods=horizon)
        future_ordinals = future_dates.map(pd.Timestamp.toordinal).values.reshape(-1, 1)
        future_preds = model.predict(future_ordinals)

        server_name = group["dbserver_text"].iloc[0]
        drive_letter = group["DriveLetter_text"].iloc[0]

        forecast_df = pd.DataFrame({
            "dbserver": server_name,
            "DriveLetter": drive_letter,
            "date": future_dates,
            "predicted_PercentageFree": future_preds,
            "forecast_horizon": f"{horizon}_days"
        })
        forecast_results.append(forecast_df)

forecast_all = pd.concat(forecast_results)

# Group by horizons to get the mean projected space
grouped_forecast = forecast_all.groupby(["dbserver", "DriveLetter", "forecast_horizon"])["predicted_PercentageFree"].mean().reset_index()

# Save results locally using relative paths
grouped_forecast.to_csv("./output/grouped_forecast.csv", index=False)
grouped_forecast.head()

,dbserver,DriveLetter,forecast_horizon,predicted_PercentageFree
0,SQLSAMPLE-0001,C,30_days,85.073522
1,SQLSAMPLE-0001,C,365_days,84.859636
2,SQLSAMPLE-0001,C,60_days,85.054368
3,SQLSAMPLE-0001,C,90_days,85.035214
4,SQLSAMPLE-0001,D,30_days,67.444174


## 5. Visualizing the "Danger Zone" (< 30% Free Space)
Filter the forecast down to drives that are predicted to fall below our 30% threshold and chart them.

In [33]:
# Filter rows where predicted free space is less than 30%
filtered_forecast = grouped_forecast[grouped_forecast["predicted_PercentageFree"] < 30].copy()

# Combine server and drive for labeling (FIXED: Removed .loc to properly update column)
filtered_forecast["DriveLetter"] = filtered_forecast["dbserver"] + " - Drive " + filtered_forecast["DriveLetter"]

# Set custom order for forecast_horizon
filtered_forecast["forecast_horizon"] = pd.Categorical(
    filtered_forecast["forecast_horizon"],
    categories=["30_days", "60_days", "90_days", "365_days"],
    ordered=True
)

filtered_forecast = filtered_forecast.sort_values(["forecast_horizon", "predicted_PercentageFree"])

# Create a bar plot
fig = px.bar(
    filtered_forecast,
    x="forecast_horizon",
    y="predicted_PercentageFree",
    color="dbserver",
    barmode="group",
    title="Forecasted Drive Space < 30% Free by Server and Drive",
    labels={
        "predicted_PercentageFree": "% Free",
        "dbserver": "Server & Drive",
        "forecast_horizon": "Forecast Horizon"    
    }
)

# Export filtered data and image
filtered_forecast.to_csv("./output/forecast_below_30_labeled.csv", index=False)
# fig.write_image("./output/forecast_below_30_labeled.png")

# Display the interactive plot in the notebook
fig.show()